# 실행 환경 확인

워크스페이스의 파이썬, CUDA, GPU가 정상인지 확인하는 노트북입니다. 위에서부터 차례로 실행하세요.

## 1. 파이썬과 시스템

In [1]:
import os
import platform
import sys

# 세 프레임워크가 한 GPU를 나눠 쓰도록 메모리 선점을 끈다 (이미지에 이미 설정돼 있으면 그대로 둔다)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ.setdefault("KERAS_BACKEND", "jax")

print("Python  :", sys.version)
print("실행 파일:", sys.executable)
print("OS      :", platform.platform())
print("CPU 코어:", os.cpu_count())

Python  : 3.12.16 (main, Nov  5 2026, 07:58:20) [GCC 14.2.0]
실행 파일: /usr/local/bin/python
OS      : Linux-5.15.0-86-generic-x86_64-with-glibc2.41
CPU 코어: 16


## 2. GPU와 드라이버

`nvidia-smi`가 실패하면 컨테이너에 GPU가 할당되지 않은 것입니다.

In [2]:
!nvidia-smi

[Coaster NOTICE (pid:10015 thread=140017538881408 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
Mon Nov 23 06:42:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 620.70.06              KMD Version: 620.70.06     CUDA UMD Version: 13.5     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  PART_8 NVIDIA RTX A2000        On  |   00000000:03:00.0 Off |                  Off |
| 30%   24C    P8             12W /   70W |       0MiB /    

## 3. 설치된 패키지 버전

In [3]:
from importlib.metadata import PackageNotFoundError, version

packages = [
    "tensorflow", "torch", "jax", "jaxlib",
    "keras", "keras-hub", "keras-tuner",
    "numpy", "matplotlib", "jupyterlab",
]
for name in packages:
    try:
        print(f"{name:12s} {version(name)}")
    except PackageNotFoundError:
        print(f"{name:12s} (설치되지 않음)")

tensorflow   2.21.0
torch        2.15.0
jax          0.12.1
jaxlib       0.12.1
keras        3.15.1
keras-hub    0.33.0
keras-tuner  1.4.8
numpy        2.5.5
matplotlib   3.11.2
jupyterlab   4.6.6


## 4. PyTorch

In [4]:
import torch

print("torch       :", torch.__version__)
print("CUDA (빌드) :", torch.version.cuda)
print("cuDNN       :", torch.backends.cudnn.version())
print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}, {props.total_memory / 1024**3:.1f} GB")
    x = torch.rand(2000, 2000, device="cuda")
    print("행렬 곱 결과 장치:", (x @ x).device)

[Coaster NOTICE (pid:9943 thread=139776553387904 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)


torch       : 2.15.0+cu132
CUDA (빌드) : 13.2
cuDNN       : 92801
GPU 사용 가능: True
GPU 0: NVIDIA RTX A2000, 0.5 GB


[Coaster NOTICE (pid:9943 thread=139776553387904 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster NOTICE (pid:9943 thread=139776553387904 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster NOTICE (pid:9943 thread=139776553387904 comm.c:387)]: block_container_agent: resolve_host_pid response: host_pid=3408816
[Coaster NOTICE (pid:9943 thread=139776553387904 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster NOTICE (pid:9943 thread=139776553387904 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster NOTICE (pid:9943 thread=139776553387904 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster NOTICE

RuntimeError: CUDA error: CUBLAS_STATUS_INVALID_VALUE when calling `cublasGemmEx( handle, opa, opb, m, n, k, &alpha, a, CUDA_R_32F, lda, b, CUDA_R_32F, ldb, &beta, c, CUDA_R_32F, ldc, compute_type, CUBLAS_GEMM_DEFAULT_TENSOR_OP)`

[Coaster NOTICE (pid:9943 thread=139776553387904 libvgpu.c:501)]: CTX_EXIT_METHOD=sync: cuCtxSynchronize (pid=9943, ctx=0x55d816780690)
[Coaster NOTICE (pid:9943 thread=139776553387904 libvgpu.c:504)]: CTX_EXIT_METHOD=sync: cuCtxSynchronize done (pid=9943, res=0)


## 5. TensorFlow

In [ ]:
import tensorflow as tf

build = tf.sysconfig.get_build_info()
print("tensorflow  :", tf.__version__)
print("CUDA (빌드) :", build.get("cuda_version"))
print("cuDNN (빌드):", build.get("cudnn_version"))

gpus = tf.config.list_physical_devices("GPU")
print("GPU 목록    :", gpus)

if gpus:
    x = tf.random.uniform((2000, 2000))
    print("행렬 곱 결과 장치:", tf.matmul(x, x).device)

## 6. JAX

In [ ]:
import jax
import jax.numpy as jnp

print("jax      :", jax.__version__)
print("장치 목록:", jax.devices())

x = jax.random.uniform(jax.random.key(0), (2000, 2000))
print("행렬 곱 결과 장치:", (x @ x).devices())

## 7. 케라스로 간단한 모델 훈련

무작위 데이터로 작은 모델을 한 번 훈련해 전체 흐름이 동작하는지 확인합니다.

In [ ]:
import keras
import numpy as np

print("keras  :", keras.__version__)
print("백엔드 :", keras.backend.backend())

x_train = np.random.rand(2000, 20).astype("float32")
y_train = (x_train.sum(axis=1) > 10).astype("float32")

model = keras.Sequential([
    keras.Input(shape=(20,)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history = model.fit(x_train, y_train, epochs=5, batch_size=64, verbose=2)